### breast_cancer.py

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay
import warnings

data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Task (a): Confusion Matrix (threshold=0.5) ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("\nInterpretation:")
print(f"True Negatives (TN):  {cm[0,0]} - Correctly predicted benign")
print(f"False Positives (FP): {cm[0,1]} - Benign predicted as malignant")
print(f"False Negatives (FN): {cm[1,0]} - Malignant predicted as benign (CRITICAL ERROR)")
print(f"True Positives (TP):  {cm[1,1]} - Correctly predicted malignant")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))

print("\n=== Task (b): Threshold = 0.3 ===")
y_proba = model.predict_proba(X_test)[:, 1]
y_pred_03 = (y_proba >= 0.3).astype(int)
cm_03 = confusion_matrix(y_test, y_pred_03)
print(cm_03)
print("\nClassification Report (threshold=0.3):")
print(classification_report(y_test, y_pred_03, target_names=["malignant", "benign"]))

recall_05_mal = cm[0,0] / (cm[0,0] + cm[0,1])
recall_03_mal = cm_03[0,0] / (cm_03[0,0] + cm_03[0,1])
recall_05_ben = cm[1,1] / (cm[1,0] + cm[1,1])
recall_03_ben = cm_03[1,1] / (cm_03[1,0] + cm_03[1,1])
print(f"\nRecall comparison:")
print(f"  Malignant (class 0): {recall_05_mal:.4f} -> {recall_03_mal:.4f} (change: {recall_03_mal - recall_05_mal:+.4f})")
print(f"  Benign (class 1):    {recall_05_ben:.4f} -> {recall_03_ben:.4f} (change: {recall_03_ben - recall_05_ben:+.4f})")
print("-> Lower threshold = more predictions as positive (benign=1), increasing benign recall.")

print("\n=== Task (c): max_iter=10 (very low) ===")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    model_low = LogisticRegression(max_iter=10, random_state=42)
    model_low.fit(X_train, y_train)
    if len(w) > 0:
        print(f"Warning: {w[0].message}")
        print("\nMeaning: The optimizer did not converge in 10 iterations.")
        print("-> Model coefficients are not optimal. Increase max_iter for better results.")
    else:
        print("No warning (unexpected - may need more features or different solver)")

print("\n=== Task (d): ROC Curve ===")
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_title("ROC Curve - Breast Cancer Classification")
ax.legend()
plt.tight_layout()
plt.savefig("roc_curve_breast_cancer.png", dpi=150)
plt.show()
print("ROC curve saved as 'roc_curve_breast_cancer.png'")
